# Recycling Center Alloy Problem with AMPL in Python
## AMPL Modeling for Problem 7

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

```python
%pip install amplpy
```

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
from amplpy import AMPL, modules

modules.install('highs')


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

## Problem: Recycling Center Alloy Blend

An industrial recycling center uses two aluminum scraps (A and B) to produce a special alloy.

**Scrap costs:**
- Scrap A: $100/ton
- Scrap B: $80/ton

**Scrap composition:**

| Element   | Scrap A | Scrap B |
|-----------|---------|--------|
| Aluminum  | 6%      | 3%     |
| Silicon   | 3%      | 6%     |
| Carbon    | 4%      | 3%     |

**Alloy specifications (for 1,000 tons):**
- Aluminum: 3% to 6%
- Silicon: 3% to 5%
- Carbon: 3% to 7%

Determine the optimal scrap mix to produce 1,000 tons of alloy at minimum cost.

In [4]:
SCRAP_DATA = {
    "A": {"cost": 100, "aluminum": 0.06, "silicon": 0.03, "carbon": 0.04},
    "B": {"cost": 80, "aluminum": 0.03, "silicon": 0.06, "carbon": 0.03},
}

SPECS = {
    "aluminum": {"min": 0.03, "max": 0.06},
    "silicon": {"min": 0.03, "max": 0.05},
    "carbon": {"min": 0.03, "max": 0.07},
}

TOTAL = 1000

In [5]:
@timed
def solve_recycling(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        var xA >= 0;
        var xB >= 0;

        minimize TotalCost:
            100 * xA + 80 * xB;

        subject to TotalProduction:
            xA + xB = 1000;

        subject to AluminumMin:
            0.06 * xA + 0.03 * xB >= 0.03 * 1000;

        subject to AluminumMax:
            0.06 * xA + 0.03 * xB <= 0.06 * 1000;

        subject to SiliconMin:
            0.03 * xA + 0.06 * xB >= 0.03 * 1000;

        subject to SiliconMax:
            0.03 * xA + 0.06 * xB <= 0.05 * 1000;

        subject to CarbonMin:
            0.04 * xA + 0.03 * xB >= 0.03 * 1000;

        subject to CarbonMax:
            0.04 * xA + 0.03 * xB <= 0.07 * 1000;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "xA": round(ampl.get_value("xA"), 4),
        "xB": round(ampl.get_value("xB"), 4),
        "cost": round(ampl.get_value("TotalCost"), 4),
    }
    return solution

In [6]:
result, elapsed = solve_recycling()

print("=== RECYCLING CENTER RESULTS ===")
print(f"Scrap A -> {result['xA']:.2f} tons ({result['xA']/10:.2f}%)")
print(f"Scrap B -> {result['xB']:.2f} tons ({result['xB']/10:.2f}%)")
print(f"Minimum cost -> ${result['cost']:,.2f}")
print(f"Time         -> {elapsed:.8f} seconds")
print()

xA, xB = result["xA"], result["xB"]
al = 0.06*xA + 0.03*xB
si = 0.03*xA + 0.06*xB
ca = 0.04*xA + 0.03*xB

print("Alloy composition verification:")
print(f"  Aluminum: {al:.2f} tons ({al/10:.2f}%) [spec: 3-6%]")
print(f"  Silicon:  {si:.2f} tons ({si/10:.2f}%) [spec: 3-5%]")
print(f"  Carbon:   {ca:.2f} tons ({ca/10:.2f}%) [spec: 3-7%]")

HiGHS 1.11.0=== RECYCLING CENTER RESULTS ===
Scrap A -> 333.33 tons (33.33%)
Scrap B -> 666.67 tons (66.67%)
Minimum cost -> $86,666.67
Time         -> 0.01034697 seconds

Alloy composition verification:
  Aluminum: 40.00 tons (4.00%) [spec: 3-6%]
  Silicon:  50.00 tons (5.00%) [spec: 3-5%]
  Carbon:   33.33 tons (3.33%) [spec: 3-7%]


## Sensitivity Analysis

In [7]:
ampl = create_ampl_instance()

ampl.eval(
    r'''
    var xA >= 0;
    var xB >= 0;

    minimize TotalCost:
        100 * xA + 80 * xB;

    subject to TotalProduction:
        xA + xB = 1000;

    subject to AluminumMin:
        0.06 * xA + 0.03 * xB >= 30;

    subject to AluminumMax:
        0.06 * xA + 0.03 * xB <= 60;

    subject to SiliconMin:
        0.03 * xA + 0.06 * xB >= 30;

    subject to SiliconMax:
        0.03 * xA + 0.06 * xB <= 50;

    subject to CarbonMin:
        0.04 * xA + 0.03 * xB >= 30;

    subject to CarbonMax:
        0.04 * xA + 0.03 * xB <= 70;
    '''
)
ampl.solve()

constraints = ["TotalProduction", "AluminumMin", "AluminumMax",
               "SiliconMin", "SiliconMax", "CarbonMin", "CarbonMax"]

print("=== SHADOW PRICES AND SLACK ===")
print(f"{'Constraint':<20} {'Shadow Price':>14} {'Slack':>10}")
print("-" * 46)
for c in constraints:
    con = ampl.get_constraint(c)
    print(f"{c:<20} {con.dual():>14.4f} {con.slack():>10.4f}")

print()
print("=== REDUCED COSTS ===")
print(f"xA  RC = {ampl.get_variable('xA').rc():.4f}  Value = {ampl.get_variable('xA').value():.2f}")
print(f"xB  RC = {ampl.get_variable('xB').rc():.4f}  Value = {ampl.get_variable('xB').value():.2f}")

print()
print("=== BINDING CONSTRAINTS ===")
for c in constraints:
    con = ampl.get_constraint(c)
    if abs(con.slack()) < 1e-6:
        print(f"  {c} is BINDING (shadow price = {con.dual():.4f})")

HiGHS 1.11.0=== SHADOW PRICES AND SLACK ===
Constraint             Shadow Price      Slack
----------------------------------------------
TotalProduction            120.0000     0.0000
AluminumMin                  0.0000    10.0000
AluminumMax                  0.0000    20.0000
SiliconMin                   0.0000    20.0000
SiliconMax                -666.6667     0.0000
CarbonMin                    0.0000     3.3333
CarbonMax                    0.0000    36.6667

=== REDUCED COSTS ===
xA  RC = 0.0000  Value = 333.33
xB  RC = 0.0000  Value = 666.67

=== BINDING CONSTRAINTS ===
  TotalProduction is BINDING (shadow price = 120.0000)
  SiliconMax is BINDING (shadow price = -666.6667)


## Interpretation

The recycling center alloy problem reveals:

- **Total production constraint**: The shadow price shows the marginal cost of producing one additional ton of alloy.
- **Composition bounds**: Binding composition constraints indicate which element specifications most affect cost. Scrap A is richer in aluminum and carbon, while Scrap B has more silicon.
- **Cost trade-off**: Scrap B is cheaper ($80 vs $100) but has different element ratios. The optimal blend balances cost against composition requirements.
- **Non-binding bounds**: If a composition bound is not binding, relaxing it would not change the solution — the optimal mix already satisfies it with margin.